### Retriever And Chain With Langchain

In [1]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("MS_Resume_Ausaf.pdf")
docs = loader.load()
docs

[Document(page_content='MOHAMMAD AUSAF SHAH\n+91-9682138656 | ausafshah18@gmail.com | Linkdin | Github | Portfolio\nEDUCA TION\nSRM Institute of Science and T echnology September 2020 – May 2024\nB.Tech in Computer Science and Engineering - CGPA 9.09/10 Kattankulathur, Tamil Nadu, India\nRelevant Coursework: Data Structures & Algorithms, Operating Systems, Object Oriented Programming, Database\nManagement Systems, Computer Networks\nBurn Hall School 2019 – 2020\nClass XII - 92.2% Srinagar, J&K, India\nBurn Hall School 2017 – 2018\nClass X - 92% Srinagar, J&K, India\nEXPERIENCE\nEngineer I Software Development July 2024 – Current\nVerizon | Spring Boot, Spring Reactive, Java, React.js, Node.js, Express.js, React Native | Chennai, India\n• Developed a CSRF filter in the Spring Boot production backend, reducing CORS issues to 0%.\n• Improved business logic across multiple endpoints in the production backend.\n• Built a high-performance React Native mobile app POC for the My Verizon App, r

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
text_splitter.split_documents(docs)[:5]

[Document(page_content='MOHAMMAD AUSAF SHAH\n+91-9682138656 | ausafshah18@gmail.com | Linkdin | Github | Portfolio\nEDUCA TION\nSRM Institute of Science and T echnology September 2020 – May 2024\nB.Tech in Computer Science and Engineering - CGPA 9.09/10 Kattankulathur, Tamil Nadu, India\nRelevant Coursework: Data Structures & Algorithms, Operating Systems, Object Oriented Programming, Database\nManagement Systems, Computer Networks\nBurn Hall School 2019 – 2020\nClass XII - 92.2% Srinagar, J&K, India\nBurn Hall School 2017 – 2018\nClass X - 92% Srinagar, J&K, India\nEXPERIENCE\nEngineer I Software Development July 2024 – Current\nVerizon | Spring Boot, Spring Reactive, Java, React.js, Node.js, Express.js, React Native | Chennai, India\n• Developed a CSRF filter in the Spring Boot production backend, reducing CORS issues to 0%.\n• Improved business logic across multiple endpoints in the production backend.\n• Built a high-performance React Native mobile app POC for the My Verizon App, r

In [3]:
documents=text_splitter.split_documents(docs)
documents

[Document(page_content='MOHAMMAD AUSAF SHAH\n+91-9682138656 | ausafshah18@gmail.com | Linkdin | Github | Portfolio\nEDUCA TION\nSRM Institute of Science and T echnology September 2020 – May 2024\nB.Tech in Computer Science and Engineering - CGPA 9.09/10 Kattankulathur, Tamil Nadu, India\nRelevant Coursework: Data Structures & Algorithms, Operating Systems, Object Oriented Programming, Database\nManagement Systems, Computer Networks\nBurn Hall School 2019 – 2020\nClass XII - 92.2% Srinagar, J&K, India\nBurn Hall School 2017 – 2018\nClass X - 92% Srinagar, J&K, India\nEXPERIENCE\nEngineer I Software Development July 2024 – Current\nVerizon | Spring Boot, Spring Reactive, Java, React.js, Node.js, Express.js, React Native | Chennai, India\n• Developed a CSRF filter in the Spring Boot production backend, reducing CORS issues to 0%.\n• Improved business logic across multiple endpoints in the production backend.\n• Built a high-performance React Native mobile app POC for the My Verizon App, r

In [4]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS # it's kind of a vectorStore developed by meta

#db=FAISS.from_documents(documents[:30],OpenAIEmbeddings()) # error expected here as I don't have subscription

In [5]:
# Use the dedicated embedding model for the database
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(documents[:30], embeddings)

In [6]:
db

In [7]:
query = "Developed a CSRF filter in the Spring Boot"

result = db.similarity_search(query)
result[0].page_content

'MOHAMMAD AUSAF SHAH\n+91-9682138656 | ausafshah18@gmail.com | Linkdin | Github | Portfolio\nEDUCA TION\nSRM Institute of Science and T echnology September 2020 – May 2024\nB.Tech in Computer Science and Engineering - CGPA 9.09/10 Kattankulathur, Tamil Nadu, India\nRelevant Coursework: Data Structures & Algorithms, Operating Systems, Object Oriented Programming, Database\nManagement Systems, Computer Networks\nBurn Hall School 2019 – 2020\nClass XII - 92.2% Srinagar, J&K, India\nBurn Hall School 2017 – 2018\nClass X - 92% Srinagar, J&K, India\nEXPERIENCE\nEngineer I Software Development July 2024 – Current\nVerizon | Spring Boot, Spring Reactive, Java, React.js, Node.js, Express.js, React Native | Chennai, India\n• Developed a CSRF filter in the Spring Boot production backend, reducing CORS issues to 0%.\n• Improved business logic across multiple endpoints in the production backend.\n• Built a high-performance React Native mobile app POC for the My Verizon App, reducing page load time 

In [8]:
from langchain_community.llms import Ollama
## Load Ollama glm-4.6:cloud LLM model
llm=Ollama(model="glm-4.6:cloud")
llm

Ollama(model='glm-4.6:cloud')

In [9]:
## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context. 
Think step by step before providing a detailed answer. 
I will tip you $1000 if the user finds the answer helpful. 
<context>
{context}
</context>
Question: {input}""")

In [10]:
## Chain Introduction ********
## Create Stuff Docment Chain

from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm,prompt)

In [11]:
### Retriever introduction *****

"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to 
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well. 
 https://python.langchain.com/docs/modules/data_connection/retrievers/   
"""

retriever=db.as_retriever() # We connected our vectorStore db to an interface (i.e retriever)
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001ADFE012450>)

In [14]:
###  Retriever Chain ***

"""
Retrieval chain:This chain takes in a user inquiry, which is then
passed to the retriever to fetch relevant documents. Those documents 
(and original inputs) are then passed to an LLM to generate a response
https://python.langchain.com/docs/modules/chains/
"""
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [16]:
response = retrieval_chain.invoke({"input":"A CSRF filter in the Spring Boot"})

In [17]:
response['answer']

'**Step-by-step thinking:**\n\n1.  I will scan the provided context (the resume) for the keywords "CSRF filter" and "Spring Boot".\n2.  I have located these keywords in the "EXPERIENCE" section, specifically under the job title "Engineer I Software Development" at Verizon.\n3.  The relevant bullet point states: "• Developed a CSRF filter in the Spring Boot production backend, reducing CORS issues to 0%."\n4.  This is the only piece of information in the context that directly addresses the query. I will formulate my answer using only this sentence, without adding any external knowledge about what CSRF or CORS are.\n5.  The answer should state what was done and the result, as described in the context.\n\n**Answer:**\n\nBased on the provided context, a CSRF filter was developed in the Spring Boot production backend. This development resulted in reducing CORS issues to 0%.'